# Project 1: Brain Tumor Classification using CNN (ResNet18)

This notebook performs complete brain tumor classification from MRI images using a pretrained ResNet18 CNN.
The dataset ZIP will be uploaded manually and extracted automatically inside runtime.

Github Repo Link: https://github.com/muhammadhaseeba951-code/Brain-Tumor-Classification


## Step 1: Import Libraries

We import all required libraries:
- OpenCV for image handling
- PyTorch for CNN training
- Sklearn for evaluation


In [ ]:
import os
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models


## Step 2: Extract Dataset ZIP

Manually upload dataset file: `brain_tumor_dataset.zip`. The ZIP file is extracted automatically into runtime.



In [ ]:
zip_path = '/content/brain_tumor_dataset.zip'
extract_path = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('Dataset extracted successfully')
print(os.listdir('/content'))


FileNotFoundError: [Errno 2] No such file or directory: '/content/brain_tumor_dataset.zip'

## Step 3: Load Dataset

After extraction, the dataset folder is loaded.
The dataset contains:
- yes (tumor)
- no (non-tumor)


In [ ]:
dataset_path = '/content/brain_tumor_dataset'
print('Dataset folder:', os.listdir(dataset_path))


## Step 4: Preprocess Images

Images are resized and normalized to match ResNet18 input requirements.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
print('Classes:', dataset.classes)


## Step 5: Explore Dataset

Display sample MRI images to verify correct loading.


In [ ]:
fig, axes = plt.subplots(1,5, figsize=(15,5))
for i in range(5):
    img_path, label = dataset.samples[i]
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(dataset.classes[label])
    axes[i].axis('off')
plt.show()


## Step 6: Split Dataset

We split data into training and testing sets.


In [ ]:
indices = list(range(len(dataset)))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=dataset.targets,
    random_state=42
)
train_dataset = Subset(dataset, train_idx)
test_dataset = Subset(dataset, test_idx)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


## Step 7: Load ResNet18 CNN

ResNet18 is a pretrained CNN used for transfer learning.
Only the final layer is trained.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


## Step 8: Define Loss and Optimizer

Loss measures error. Optimizer improves the model.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


## Step 9: Train Model

The model learns patterns from MRI images.


In [ ]:
epochs = 5
train_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f'Epoch {epoch+1}/{epochs}: {epoch_loss:.4f}')


## Step 10: Evaluate Model

We calculate accuracy, precision, and recall.


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))


## Step 11: Confusion Matrix

Shows model prediction performance visually.


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d')
plt.show()


## Step 12: Training Loss Graph

Shows how loss decreases during training.


In [ ]:
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()


## Step 13: Save and Download Model

The trained model is saved in runtime and automatically downloaded.


In [ ]:
torch.save(model.state_dict(), 'brain_tumor_model.pth')
print('Model saved in runtime')

from google.colab import files
files.download('brain_tumor_model.pth')
